# Day 1 | Notebook 0: Docker Infrastructure & Design
**Author: Sattaya Singkul**

Before we program our database, we must design the environment. We are building a **Multi-Container Architecture**. 

### 1. Docker Design: Orchestration
In production, you never run your Database and your Application on the same "machine" or container. Why?
*   **Isolation**: If the app crashes (OOM), the DB stays alive.
*   **Scalability**: You can scale to 10 App servers while keeping 1 DB server.

We use **Docker Compose** to orchestrate these two services. Look at your `docker-compose.yml` file. You will see two services defined: `notebook-env` (The Brain) and `redis-vector-db` (The Muscle).

---

### Phase 1: Images vs. Containers (The Practice)
A common mistake is confusing an **Image** (The Template) with a **Container** (The running process).

**Exercise 1.1: Identify the Template**
Run the cell below to see the images currently registered on your machine.


In [ ]:
# '!' allows us to run terminal commands directly from the notebook
!docker image ls | grep -E 'redis|jupyter'

**Exercise 1.2: Identify the Running Process**
Now, let's see which "Box" is actually active and consuming RAM right now.


In [ ]:
!docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}" | grep vector-db

---

### Phase 2: Virtual Networking & Internal DNS
Docker creates a private "Virtual LAN". Inside this network, containers don't use IP addresses; they use **Service Names**. 

**Exercise 2.1: Resolving Hostnames**
Let's prove that this notebook (The Brain) can find the "Muscle" using only its name. This is called **Internal DNS**.


In [ ]:
import socket

def resolve_service(name):
    try:
        ip = socket.gethostbyname(name)
        print(f"✅ DNS Resolved: '{name}' is at internal IP {ip}")
    except Exception as e:
        print(f"❌ DNS Failed: Could not find '{name}'. Is the container running?")

resolve_service("redis-vector-db")


---

### Phase 3: Volumes & Data Persistence
Containers are "Ephemeral" (Temporary). If you delete a container, the data inside is gone.
**Volumes** solve this by "mounting" your laptop's folder into the container.

**Exercise 3.1: The Persistence Test**
We will create a file from *inside* this container, and you can verify it exists on your *actual laptop* right now.


In [ ]:
import os

test_file = "/home/jovyan/work/docker_persistence_test.txt"

with open(test_file, "w") as f:
    f.write("Hello from the Docker Container! This file is now saved on your physical hard drive.")

print(f"✅ File created at {test_file}")
print("Check your local folder on your computer. You should see 'docker_persistence_test.txt' there!")


---

### Phase 4: Lifecycle & Logs (Debugging)
In engineering, reading logs is the most important skill. If the Vector DB isn't responding, we check the logs.

**Exercise 4.1: Checking Redis Heartbeat**
Let's use the Docker CLI to read the last 5 lines of the Redis startup log.


In [ ]:
# Read logs from the 'Muscle' container
!docker logs --tail 5 vector-db-course-redis

### Phase 5: The Connection Check
Now that we understand the Principles (Images, Network, Volumes), we can confirm the final connection.


In [ ]:
def check_connection(host, port):
    try:
        socket.create_connection((host, port), timeout=3)
        print(f"✅ READY: Vector Database at '{host}:{port}' is reachable.")
        print("You have successfully mastered Docker Principles for Vector Engineering.")
    except Exception as e:
        print(f"❌ OFFLINE: Cannot reach '{host}'. Check your docker-compose logs.")

check_connection("redis-vector-db", 6379)


🎉 **Congratulations!**
You have verified your architecture. You understand:
1. How images become running containers.
2. How containers talk via Service Names (DNS).
3. How Data persists via Volumes.
4. How to debug using Logs.

Proceed to `foundation/Notebook_1_Math.ipynb` to start building your first Vector Search engine!
